# NB07 — Manuscript Asset Assembly
## BAREKENG MS 21575 — Camera-Ready Packaging

**Stage overview.** The final stage. Computes nothing. Validates all six upstream
handshakes, assembles every figure and table into `outputs/manuscript_ready/` under
manuscript-facing names, emits LaTeX for the four core tables, reconciles the inventory
against what the pipeline claims to have produced, and prints an end-to-end parity summary.

| | |
|---|---|
| **Manuscript link** | All figures and tables |
| **Upstream** | NB01–NB06, all six handshakes |
| **Downstream** | the manuscript itself |

### Why this notebook recomputes nothing

Every number in the revised paper must be traceable to exactly one notebook. If NB07
recalculated anything, a discrepancy between it and its source would be undetectable —
there would be two candidate truths. It therefore only reads, renames, reformats and
checks.

### Inventory reconciliation

Cell 07 compares what exists on disk against what the handshakes claim. Two failure modes
matter:

- **Missing** — a handshake names an artefact that is not present. The pipeline is
  incomplete and the manuscript would cite something that does not exist.
- **Orphaned** — a file exists that no handshake claims. Usually a leftover from a
  superseded run. These are the dangerous ones: they look like valid outputs and can be
  pasted into a manuscript long after the code that produced them has changed.

In [1]:
# Cell 01 — Mount Storage & Define Paths

import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PROJECT_FOLDER_NAME = "BarekengPaper1-Revision"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = Path(f"/content/drive/MyDrive/{PROJECT_FOLDER_NAME}")
else:
    BASE = Path.cwd()
    if BASE.name == "notebooks":
        BASE = BASE.parent

PROCESSED = BASE / "data" / "processed"
FIGURES = BASE / "outputs" / "figures"
TABLES = BASE / "outputs" / "tables"
EXPORTS = BASE / "outputs" / "notebook_exports"
MANUSCRIPT = BASE / "outputs" / "manuscript_ready"
MS_FIGS = MANUSCRIPT / "figures"
MS_TABS = MANUSCRIPT / "tables"

for d in [MANUSCRIPT, MS_FIGS, MS_TABS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local workstation'}")
print(f"Base        : {BASE}")
print(f"Destination : {MANUSCRIPT.relative_to(BASE)}")

Mounted at /content/drive
Environment : Google Colab
Base        : /content/drive/MyDrive/BarekengPaper1-Revision
Destination : outputs/manuscript_ready


In [2]:
# Cell 02 — Imports & Environment Capture

import json
import platform
from datetime import datetime

import pandas as pd

ENVIRONMENT = {"python": platform.python_version(), "pandas": pd.__version__}

print("NB07 performs no computation; only pandas is required for table reformatting.\n")
for pkg, ver in ENVIRONMENT.items():
    print(f"  {pkg:<10} {ver}")

NB07 performs no computation; only pandas is required for table reformatting.

  python     3.13.15
  pandas     2.2.3


In [3]:
# Cell 03 — Validate All Six Upstream Handshakes

REQUIRED = ["NB01", "NB02", "NB03", "NB04", "NB05", "NB06"]
handshakes = {}
problems = []

for nb in REQUIRED:
    path = EXPORTS / f"summary_{nb}.json"
    if not path.exists():
        problems.append(f"{nb}: handshake missing ({path.name})")
        continue
    with open(path, "r", encoding="utf-8") as fh:
        data = json.load(fh)
    handshakes[nb] = data
    if data.get("status") != "SUCCESS":
        problems.append(f"{nb}: status is {data.get('status')!r}, not SUCCESS")
    failed = [k for k, v in data.get("assertions", {}).items() if not v]
    if failed:
        problems.append(f"{nb}: assertions failed -> {failed}")

if problems:
    raise AssertionError(
        "Upstream pipeline is not clean; refusing to assemble manuscript assets:\n  "
        + "\n  ".join(problems)
    )

print(f"{'notebook':<10}{'ran':<22}{'assertions':<12}{'comment'}")
print("-" * 62)
for nb in REQUIRED:
    d = handshakes[nb]
    n_assert = len(d.get("assertions", {}))
    print(f"{nb:<10}{d['timestamp'][:19]:<22}{str(n_assert) + ' passed':<12}"
          f"{d.get('addresses_comment', '-')}")
print(f"\nAll {len(REQUIRED)} handshakes verified SUCCESS with every assertion passing.")

notebook  ran                   assertions  comment
--------------------------------------------------------------
NB01      2026-09-20T21:41:19   5 passed    -
NB02      2026-09-20T21:09:53   7 passed    -
NB03      2026-09-20T21:36:40   4 passed    C2
NB04      2026-09-20T22:01:48   5 passed    E1 (plus manuscript Table 2, Figures 6-7)
NB05      2026-09-20T22:28:54   6 passed    C1
NB06      2026-09-20T22:38:36   5 passed    C3

All 6 handshakes verified SUCCESS with every assertion passing.


---
## Section 1 — Asset Manifest

In [4]:
# Cell 04 — Load: Build the Manuscript Asset Manifest
# Maps each pipeline artefact to its manuscript-facing identity. Figures retaining their
# submitted numbering keep it; new material introduced by the revision is numbered from 8.

FIGURE_MANIFEST = [
    # (source file, manuscript name, section, status, comment)
    ("nb01_fig1_distributions.png", "Figure_1_distributions.png", "2.1", "revised",
     "rebuilt - submitted version was an unreadable dense bar chart"),
    ("nb01_fig2_correlation_heatmap.png", "Figure_2_correlation_heatmap.png", "2.1", "reproduced", "-"),
    ("nb02_fig4_if_scatter.png", "Figure_4_if_scatter.png", "3.2", "revised", "log-log rescaled"),
    ("nb02_fig5_lof_scatter.png", "Figure_5_lof_scatter.png", "3.2", "revised", "log-log rescaled"),
    ("nb04_fig6_venn.png", "Figure_6_venn.png", "3.3", "reproduced", "-"),
    ("nb04_fig7_contingency.png", "Figure_7_contingency.png", "3.3", "corrected",
     "submitted version's cells implied 1,000 LOF anomalies; irreconcilable with Table 2"),
    ("nb03_fig_sensitivity_heatmap.png", "Figure_8_sensitivity_heatmap.png", "3.1 (new)", "new", "C2"),
    ("nb03_fig_stability_curves.png", "Figure_9_stability_curves.png", "3.1 (new)", "new", "C2"),
    ("nb05_fig_detection_by_mechanism.png", "Figure_10_synthetic_detection.png", "3.1 (new)", "new", "C1"),
    ("nb05_fig_pr_curves.png", "Figure_11_pr_curves.png", "3.1 (new)", "new", "C1"),
    ("nb04_fig_null_model.png", "Figure_12_jaccard_null_model.png", "3.3", "new", "E1"),
    ("nb06_fig_effect_sizes.png", "Figure_13_effect_sizes.png", "3.4", "new", "C3"),
]

TABLE_MANIFEST = [
    ("nb02_table1_anomaly_counts.csv", "Table_1_anomaly_counts", "3.1", "reproduced", "-", True),
    ("nb04_table2_jaccard_overlap.csv", "Table_2_jaccard_overlap", "3.3", "reproduced", "-", True),
    ("nb06_table3_mannwhitney_effectsize.csv", "Table_3_mannwhitney_effectsize", "3.4", "extended",
     "original U values retained; rank-biserial and CLES added (C3)", True),
    ("nb06_table4_chisquare_effectsize.csv", "Table_4_chisquare_effectsize", "3.4", "extended",
     "original chi-square retained; Cramer's V added (C3)", True),
    ("nb03_parameter_justification.csv", "Table_5_parameter_justification", "3.1 (new)", "new", "C2", False),
    ("nb03_sensitivity_if.csv", "Table_6_sensitivity_if", "3.1 (new)", "new", "C2", False),
    ("nb03_sensitivity_lof.csv", "Table_7_sensitivity_lof", "3.1 (new)", "new", "C2", False),
    ("nb05_synthetic_validation.csv", "Table_8_synthetic_validation", "3.1 (new)", "new", "C1", False),
    ("nb05_ranking_metrics.csv", "Table_9_ranking_metrics", "3.1 (new)", "new", "C1", False),
    ("nb05_injection_rate_sensitivity.csv", "Table_10_injection_rate_sensitivity", "3.1 (new)", "new",
     "C1 - methodological caveat on evaluating density-based detectors", False),
    ("nb04_jaccard_null_model.csv", "Table_11_jaccard_null_model", "3.3", "new", "E1", False),
    ("nb04_jaccard_bootstrap_ci.csv", "Table_12_jaccard_bootstrap_ci", "3.3", "new", "E1", False),
    ("nb05_consensus_case_profiles.csv", "Table_13_consensus_case_profiles", "3.2 (new)", "new",
     "C1 - descriptive only, NOT expert validation", False),
    ("nb01_descriptive_statistics.csv", "Table_14_descriptive_statistics", "2.1", "new", "E1", False),
]

fig_df = pd.DataFrame(FIGURE_MANIFEST,
                      columns=["source", "manuscript_name", "section", "status", "note"])
tab_df = pd.DataFrame(TABLE_MANIFEST,
                      columns=["source", "manuscript_name", "section", "status", "note", "core"])

print(f"Figures: {len(fig_df)}  ({(fig_df['status'] == 'new').sum()} new, "
      f"{(fig_df['status'] != 'new').sum()} carried over from the submitted paper)")
print(f"Tables : {len(tab_df)}  ({(tab_df['status'] == 'new').sum()} new, "
      f"{(tab_df['status'] != 'new').sum()} carried over)")
print(f"\nCore tables (1-4, appear in the main text): {int(tab_df['core'].sum())}")

Figures: 12  (6 new, 6 carried over from the submitted paper)
Tables : 14  (10 new, 4 carried over)

Core tables (1-4, appear in the main text): 4


In [5]:
# Cell 05 — Export: Copy Figures under Manuscript Names

copied, missing = [], []
for _, r in fig_df.iterrows():
    src = FIGURES / r["source"]
    if not src.exists():
        missing.append(r["source"])
        continue
    shutil.copy2(src, MS_FIGS / r["manuscript_name"])
    copied.append(r["manuscript_name"])

if missing:
    raise FileNotFoundError(
        "Figures named in the manifest are absent from outputs/figures/:\n  "
        + "\n  ".join(missing)
        + "\nRe-run the notebook that produces them before assembling."
    )

print(f"Copied {len(copied)} figures to {MS_FIGS.relative_to(BASE)}\n")
print(fig_df[["manuscript_name", "section", "status"]].to_string(index=False))

Copied 12 figures to outputs/manuscript_ready/figures

                  manuscript_name   section     status
       Figure_1_distributions.png       2.1    revised
 Figure_2_correlation_heatmap.png       2.1 reproduced
          Figure_4_if_scatter.png       3.2    revised
         Figure_5_lof_scatter.png       3.2    revised
                Figure_6_venn.png       3.3 reproduced
         Figure_7_contingency.png       3.3  corrected
 Figure_8_sensitivity_heatmap.png 3.1 (new)        new
    Figure_9_stability_curves.png 3.1 (new)        new
Figure_10_synthetic_detection.png 3.1 (new)        new
          Figure_11_pr_curves.png 3.1 (new)        new
 Figure_12_jaccard_null_model.png       3.3        new
       Figure_13_effect_sizes.png       3.4        new


In [6]:
# Cell 06 — Export: Tables as CSV and LaTeX

copied_t, missing_t, latex_written = [], [], []

for _, r in tab_df.iterrows():
    src = TABLES / r["source"]
    if not src.exists():
        missing_t.append(r["source"])
        continue
    df = pd.read_csv(src)
    df.to_csv(MS_TABS / f"{r['manuscript_name']}.csv", index=False)
    copied_t.append(r["manuscript_name"])

    # LaTeX only for the four core tables; the rest are supplementary and stay CSV.
    if r["core"]:
        caption = r["manuscript_name"].replace("_", " ")
        tex = df.to_latex(index=False, escape=True, longtable=False,
                          caption=caption, label=f"tab:{r['manuscript_name'].lower()}")
        (MS_TABS / f"{r['manuscript_name']}.tex").write_text(tex, encoding="utf-8")
        latex_written.append(r["manuscript_name"])

if missing_t:
    raise FileNotFoundError(
        "Tables named in the manifest are absent from outputs/tables/:\n  "
        + "\n  ".join(missing_t)
    )

print(f"Copied {len(copied_t)} tables to {MS_TABS.relative_to(BASE)}")
print(f"LaTeX emitted for {len(latex_written)} core tables: {', '.join(latex_written)}\n")
print(tab_df[["manuscript_name", "section", "status", "core"]].to_string(index=False))

Copied 14 tables to outputs/manuscript_ready/tables
LaTeX emitted for 4 core tables: Table_1_anomaly_counts, Table_2_jaccard_overlap, Table_3_mannwhitney_effectsize, Table_4_chisquare_effectsize

                    manuscript_name   section     status  core
             Table_1_anomaly_counts       3.1 reproduced  True
            Table_2_jaccard_overlap       3.3 reproduced  True
     Table_3_mannwhitney_effectsize       3.4   extended  True
       Table_4_chisquare_effectsize       3.4   extended  True
    Table_5_parameter_justification 3.1 (new)        new False
             Table_6_sensitivity_if 3.1 (new)        new False
            Table_7_sensitivity_lof 3.1 (new)        new False
       Table_8_synthetic_validation 3.1 (new)        new False
            Table_9_ranking_metrics 3.1 (new)        new False
Table_10_injection_rate_sensitivity 3.1 (new)        new False
        Table_11_jaccard_null_model       3.3        new False
      Table_12_jaccard_bootstrap_ci       3.3   

---
## Section 2 — Reconciliation

In [7]:
# Cell 07 — Test: Inventory Reconciliation against the Handshakes
# Every file the pipeline produced should be claimed by exactly one handshake, and every
# claimed artefact should exist. Anything else is a leftover from a superseded run.

claimed = set()
for nb, data in handshakes.items():
    for _, rel in data.get("outputs", {}).items():
        claimed.add(Path(rel).name)

on_disk_figs = {p.name for p in FIGURES.iterdir() if p.is_file() and not p.name.startswith(".")}
on_disk_tabs = {p.name for p in TABLES.iterdir() if p.is_file() and not p.name.startswith(".")}
on_disk = on_disk_figs | on_disk_tabs

orphaned = sorted(on_disk - claimed)
claimed_missing = sorted({c for c in claimed
                          if c.endswith((".png", ".csv")) and c not in on_disk})

print(f"Artefacts claimed by handshakes : {len([c for c in claimed if c.endswith(('.png', '.csv'))])}")
print(f"Files on disk (figures + tables): {len(on_disk)}")

if claimed_missing:
    raise FileNotFoundError(
        "A handshake claims artefacts that do not exist:\n  " + "\n  ".join(claimed_missing)
    )
print("\nNo claimed artefact is missing.")

if orphaned:
    print(f"\n*** {len(orphaned)} ORPHANED FILE(S) -- present on disk, claimed by no handshake:")
    for o in orphaned:
        print(f"      {o}")
    print(
        "\n  These are not pipeline outputs. They are almost always leftovers from a superseded\n"
        "  run, and they are hazardous precisely because they look legitimate: nothing stops\n"
        "  one being pasted into the manuscript months after the code that made it changed.\n"
        "  Delete them, or move them out of outputs/, before submission.\n"
        "  They are deliberately NOT copied into manuscript_ready/."
    )
else:
    print("No orphaned files. Every artefact on disk is claimed by a handshake.")

ORPHAN_COUNT = len(orphaned)

Artefacts claimed by handshakes : 27
Files on disk (figures + tables): 29

No claimed artefact is missing.

*** 2 ORPHANED FILE(S) -- present on disk, claimed by no handshake:
      nb01_fig1_distributions.pdf
      nb01_fig2_correlation_heatmap.pdf

  These are not pipeline outputs. They are almost always leftovers from a superseded
  run, and they are hazardous precisely because they look legitimate: nothing stops
  one being pasted into the manuscript months after the code that made it changed.
  Delete them, or move them out of outputs/, before submission.
  They are deliberately NOT copied into manuscript_ready/.


In [8]:
# Cell 08 — Test: End-to-End Parity Summary
# Every published value the rebuild had to reproduce, in one place, read from the handshakes.

nb02, nb04, nb06 = handshakes["NB02"], handshakes["NB04"], handshakes["NB06"]

t3 = {(r["Comparison"], r["Feature"]): r for r in nb06["key_results"]["table3"]}
t4 = {(r["Comparison"], r["Feature"]): r for r in nb06["key_results"]["table4"]}

parity = [
    ("Table 1", "IF anomaly count", 5000, nb02["key_results"]["if_anomaly_count"], "trivial"),
    ("Table 1", "LOF(auto) anomaly count", 2565, nb02["key_results"]["lof_auto_anomaly_count"], "DIAGNOSTIC"),
    ("Table 1", "LOF(0.05) anomaly count", 5000, nb02["key_results"]["lof_05_anomaly_count"], "trivial"),
    ("Table 2", "IF n LOF(auto)", 341, nb04["key_results"]["overlap_if_lof_auto"], "DIAGNOSTIC"),
    ("Table 2", "IF n LOF(0.05)", 545, nb04["key_results"]["overlap_if_lof_05"], "DIAGNOSTIC"),
    ("Table 2", "LOF(auto) n LOF(0.05)", 2565, nb04["key_results"]["overlap_lof_auto_lof_05"], "DIAGNOSTIC"),
]
for (comp, feat), pub in [
    (("Anomaly vs Normal", "Number of Services"), 5.10e8),
    (("Anomaly vs Normal", "Average Medicare Payment Amount"), 5.36e8),
    (("IF Only vs LOF Only", "Number of Services"), 1.21e7),
    (("IF Only vs LOF Only", "Average Medicare Payment Amount"), 1.46e7),
]:
    parity.append(("Table 3", f"{comp} / {feat[:22]}", pub, t3[(comp, feat)]["U Statistic"], "DIAGNOSTIC"))
for (comp, feat), pub in [
    (("Anomaly vs Normal", "Provider Type"), 7496.33),
    (("Anomaly vs Normal", "Entity Type of the Provider"), 1855.13),
    (("IF Only vs LOF Only", "Provider Type"), 2408.72),
    (("IF Only vs LOF Only", "Entity Type of the Provider"), 157.39),
]:
    parity.append(("Table 4", f"{comp} / {feat[:22]}", pub, t4[(comp, feat)]["Chi-Square"], "DIAGNOSTIC"))

rows = []
for tbl, metric, pub, got, power in parity:
    ok = abs(float(got) - float(pub)) / abs(float(pub)) < 0.01
    rows.append({"Table": tbl, "Metric": metric, "Published": pub, "Rebuilt": got,
                 "Match": "yes" if ok else "NO", "Power": power})

parity_df = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print(parity_df.to_string(index=False))

failures = parity_df[parity_df["Match"] != "yes"]
assert failures.empty, f"End-to-end parity failure:\n{failures}"

n_diag = int((parity_df["Power"] == "DIAGNOSTIC").sum())
print(f"\nAll {len(parity_df)} published values reproduce, of which {n_diag} are diagnostic "
      f"(the remaining {len(parity_df) - n_diag} are contamination x n and hold on any input).")
ALL_PARITY_HELD = True

  Table                                       Metric    Published      Rebuilt Match      Power
Table 1                             IF anomaly count      5000.00      5000.00   yes    trivial
Table 1                      LOF(auto) anomaly count      2565.00      2565.00   yes DIAGNOSTIC
Table 1                      LOF(0.05) anomaly count      5000.00      5000.00   yes    trivial
Table 2                               IF n LOF(auto)       341.00       341.00   yes DIAGNOSTIC
Table 2                               IF n LOF(0.05)       545.00       545.00   yes DIAGNOSTIC
Table 2                        LOF(auto) n LOF(0.05)      2565.00      2565.00   yes DIAGNOSTIC
Table 3       Anomaly vs Normal / Number of Services 510000000.00 509846027.50   yes DIAGNOSTIC
Table 3   Anomaly vs Normal / Average Medicare Payme 536000000.00 536136230.50   yes DIAGNOSTIC
Table 3     IF Only vs LOF Only / Number of Services  12100000.00  12080942.50   yes DIAGNOSTIC
Table 3 IF Only vs LOF Only / Average Me

---
## Section 3 — Persistence & Handover

In [9]:
# Cell 09 — Export: Manifest, Parity Record & JSON Handshake

manifest_path = MANUSCRIPT / "asset_manifest.csv"
manifest = pd.concat([
    fig_df.assign(kind="figure"),
    tab_df.drop(columns=["core"]).assign(kind="table"),
], ignore_index=True)[["kind", "manuscript_name", "source", "section", "status", "note"]]
manifest.to_csv(manifest_path, index=False)

parity_path = MANUSCRIPT / "parity_record.csv"
parity_df.to_csv(parity_path, index=False)

summary = {
    "notebook": "NB07",
    "status": "SUCCESS",
    "timestamp": pd.Timestamp.now().isoformat(),
    "environment": ENVIRONMENT,
    "computation_performed": "none - assembly, reformatting and verification only",
    "inputs_consumed": [f"outputs/notebook_exports/summary_{nb}.json" for nb in REQUIRED],
    "outputs": {
        "manuscript_ready_dir": "outputs/manuscript_ready/",
        "asset_manifest": "outputs/manuscript_ready/asset_manifest.csv",
        "parity_record": "outputs/manuscript_ready/parity_record.csv",
    },
    "key_results": {
        "figures_assembled": int(len(fig_df)),
        "tables_assembled": int(len(tab_df)),
        "core_tables_with_latex": int(tab_df["core"].sum()),
        "new_figures": int((fig_df["status"] == "new").sum()),
        "new_tables": int((tab_df["status"] == "new").sum()),
        "orphaned_files": ORPHAN_COUNT,
        "parity_values_checked": int(len(parity_df)),
        "parity_all_match": bool(ALL_PARITY_HELD),
    },
    "assertions": {
        "all_six_handshakes_success": True,
        "no_claimed_artefact_missing": True,
        "all_manifest_figures_present": True,
        "all_manifest_tables_present": True,
        "end_to_end_parity_holds": bool(ALL_PARITY_HELD),
    },
    "downstream_note": (
        "outputs/manuscript_ready/ is the submission package. Figures 1-7 keep their submitted "
        "numbering; 8-13 are new. Figure 3 from the submitted paper (Statistical Test Selection "
        "Matrix) has NO counterpart here and was not rebuilt - it conveyed no information and "
        "should be dropped, with subsequent figures renumbered, or replaced by Figure 13. "
        "Thirteen figures is a lot for one article: Figures 8-12 are strong candidates for "
        "supplementary material, keeping the main text to the core narrative. "
        "Remaining manuscript work is text-only: C4 abstract, C5 conclusion, C6 references, "
        "E2 novelty, E3 front matter, E4 editorial pass."
    ),
}

summary_path = EXPORTS / "summary_NB07.json"
with open(summary_path, "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, default=str)

failed = [k for k, v in summary["assertions"].items() if not v]
if failed:
    raise AssertionError(f"Handshake written but assertions failed: {failed}")

print(f"{manifest_path.relative_to(BASE)}")
print(f"{parity_path.relative_to(BASE)}")
print(f"{summary_path.relative_to(BASE)}")
print(f"\nAll {len(summary['assertions'])} assertions passed.")
print(f"\nPackage contents:")
print(f"  figures : {len(list(MS_FIGS.iterdir()))}")
print(f"  tables  : {len(list(MS_TABS.iterdir()))} (CSV + LaTeX for the four core tables)")
if ORPHAN_COUNT:
    print(f"\n  WARNING: {ORPHAN_COUNT} orphaned file(s) remain in outputs/ - see Cell 07.")

outputs/manuscript_ready/asset_manifest.csv
outputs/manuscript_ready/parity_record.csv
outputs/notebook_exports/summary_NB07.json

All 5 assertions passed.

Package contents:
  figures : 12
  tables  : 18 (CSV + LaTeX for the four core tables)



---
## NB07 Complete — Pipeline Finished

All six upstream notebooks verified, every published value reproduced, and the submission
package assembled in `outputs/manuscript_ready/`.

### Two judgment calls for you

**Figure 3 has no successor.** The submitted "Statistical Test Selection Matrix" was a 2×2
colour block carrying essentially no information, and nothing in the rebuilt pipeline
reproduces it. Either drop it and renumber, or let the new effect-size figure take its place
in §3.4. Doing nothing leaves a gap in the figure sequence.

**Thirteen figures is a lot for one article.** Figures 8–12 — the sensitivity sweep, the
stability curves, the synthetic-detection panels and the PR curves — are the most natural
candidates for supplementary material. The core narrative needs Figures 1, 2, 4, 5, 6, 7,
plus 12 (the Jaccard null model, which carries E1) and 13 (effect sizes, which carries C3).

### What remains — all text, no code

| ID | Work |
|---|---|
| **C4** | Abstract: add limitations and originality/value |
| **C5** | Conclusion: restructure to answer purpose, problem, limitations |
| **C6** | References: 7 pre-2016 citations, 19/24 = 79.2% against an 80% bar — add 2–3 recent ones. The C2 and C1 citations already added are all post-2016 and count |
| **E2** | Explicit contribution statement vs [17], [18], [21] |
| **E3** | Author Contributions, Funding, Acknowledgment, Declarations — all `XXX` on p.58 |
| **E4** | Editorial pass; unify anomaly / outlier / abnormality / irregularity |

Two findings from the rebuild must be written into §3.3 and disclosed in the cover letter:
the **Jaccard null-model reinterpretation** (overlap is 2.2–2.7× chance, so "low overlap
therefore disagreement" does not hold) and the **effect-size asymmetry** (the two models
differ from each other roughly twice as much as anomalies differ from normal records, on
payment amount and provider type). One correction is also needed: the submitted **Figure 7**
did not reconcile with Table 2 and has been rebuilt.